In [27]:
!pip install -q sentence-transformers

In [28]:
!pip install -q beautifulsoup4 requests tqdm

In [29]:
!pip install -q openai

In [30]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urldefrag
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np
import getpass
from openai import OpenAI

In [31]:
BASE_URL = "https://academy.lamoda.ru/"
OUTPUT_DIR = "lamoda_academy_md"



HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0 Safari/537.36"
    )
}

In [32]:
def get_soup(url):
    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

In [33]:
def collect_section_urls():
    soup = get_soup(BASE_URL)

    section_urls = []

    for link in soup.find_all("a", class_='section-item-detail-link'):
        href = link.get("href")

        if not href:
            continue

        full_url = urljoin(BASE_URL, href)
        full_url, _ = urldefrag(full_url)

        if full_url.startswith(BASE_URL + "articles/"):
            section_urls.append(full_url)

    return sorted(set(section_urls))

In [34]:
def collect_article_urls_from_section(section_url):
    soup = get_soup(section_url)

    urls = set()

    for link in soup.find_all("a"):
        href = link.get("href")

        if not href:
            continue

        full_url = urljoin(section_url, href)
        full_url, _ = urldefrag(full_url)

        if not full_url.startswith(BASE_URL + "articles/"):
            continue

        urls.add(full_url)

    return urls

In [35]:
def collect_all_article_urls():
    section_urls = collect_section_urls()

    print("Разделов найдено:", len(section_urls))

    all_urls = set()

    for section_url in section_urls:
        try:
            urls = collect_article_urls_from_section(section_url)
            all_urls.update(urls)
            time.sleep(0.3)
        except Exception as e:
            print("Не удалось обработать раздел:", section_url, e)

    return sorted(all_urls)

In [36]:
def clean_text(text):
    lines = []

    for line in text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            lines.append(line)

    return "\n\n".join(lines)


def parse_article(url):
    soup = get_soup(url)

    title_tag = soup.find("h1")
    title = title_tag.get_text(" ", strip=True) if title_tag else "Без названия"

    article_block = soup.find("div", class_="article-detail__text")

    if article_block is None:
        raise ValueError(f"Не найден блок текста статьи: {url}")

    text = article_block.get_text("\n", strip=True)
    text = clean_text(text)
    if len(text) < 100:
        raise ValueError(f"Слишком мало текста, возможно это не статья: {url}")

    return {
        "title": title,
        "url": url,
        "text": text
    }

In [37]:
def make_safe_filename(url):
    path = url.replace(BASE_URL, "")
    path = path.strip("/")
    path = path.replace("/", "__")
    path = re.sub(r"[^a-zA-Zа-яА-Я0-9_-]+", "_", path)
    if not path:
        path = "index"

    return path + ".md"


def save_article_as_md(article, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)

    filename = make_safe_filename(article["url"])
    path = os.path.join(output_dir, filename)

    content = f"""# {article["title"]}

Источник: {article["url"]}

{article["text"]}
"""

    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

    return path

In [38]:
article_urls = collect_all_article_urls()

print("Всего найдено URL:", len(article_urls))

articles = []
failed_urls = []

for url in article_urls:
    try:
        article = parse_article(url)
        path = save_article_as_md(article)
        articles.append({
            **article,
            "path": path
        })
        time.sleep(0.3)

    except Exception as e:
        failed_urls.append((url, str(e)))

print(f'Успешно сохраненных статей: {len(articles)}')
print(f'Ошиьок: {len(failed_urls)}')

Разделов найдено: 13
Всего найдено URL: 119
Успешно сохраненных статей: 37
Ошиьок: 82


In [69]:
print("Примеры пропущенных URL:")
for url, error in failed_urls[:10]:
    print(url, "->", error)

Примеры пропущенных URL:
https://academy.lamoda.ru/articles/aktsii/ -> Не найден блок текста статьи: https://academy.lamoda.ru/articles/aktsii/
https://academy.lamoda.ru/articles/api/ -> Не найден блок текста статьи: https://academy.lamoda.ru/articles/api/
https://academy.lamoda.ru/articles/api/1-vvedenie/ -> Не найден блок текста статьи: https://academy.lamoda.ru/articles/api/1-vvedenie/
https://academy.lamoda.ru/articles/api/10-fbo/ -> Не найден блок текста статьи: https://academy.lamoda.ru/articles/api/10-fbo/
https://academy.lamoda.ru/articles/api/11-notifikatsii/ -> Не найден блок текста статьи: https://academy.lamoda.ru/articles/api/11-notifikatsii/
https://academy.lamoda.ru/articles/api/12-dostavka/ -> Не найден блок текста статьи: https://academy.lamoda.ru/articles/api/12-dostavka/
https://academy.lamoda.ru/articles/api/13-vozvraty/ -> Не найден блок текста статьи: https://academy.lamoda.ru/articles/api/13-vozvraty/
https://academy.lamoda.ru/articles/api/14-voprosy-o-tovarakh/ 

Cтраницы без текстового контейнера пропускаются как не являющиеся полноценными статьями

Парсер устойчив к ошибкам, если страница не содержит ожидаемый контейнер статьи, она пропускается и логируется в failed_urls

In [39]:
print(articles[0]["title"])
print(articles[0]["url"])
print(articles[0]["text"][:2000])

Акции с дополнительной скидкой от Lamoda
https://academy.lamoda.ru/articles/aktsii/aktsii-s-dopolnitelnoy-skidkoy-ot-lamoda/
В календаре акций вы можете увидеть акции с дополнительной скидкой от Lamoda. В таких акциях к выбранной вами скидке добавляется дополнительная скидка за счет Lamoda.

Календарь акций

Акции с дополнительной скидкой от Lamoda отмечены специальной иконкой. При наведении на акцию вы увидите подробное описание акции.

При наведении на акцию:

преимущество

«дополнительная скидка от Lamoda»

выделено ярким индикатором;

отображается дата,

до которой можно добавлять товары

, чтобы на них был начислена

дополнительная скидка от Lamoda

— она совпадает со сроком удаления товаров из акции.

Информационное окно акции

В карточке акции:

преимущество «доп. скидка» также ярко выделено визуально;

указаны

точные даты

, когда в рамках акции действует доп. скидка от Lamoda.

Страница акции

На странице акции, в которую добавлены товары, дополнительная скидка от Lamoda отоб

In [40]:
documents = []

for article in articles:
    documents.append({
        "title": article["title"],
        "url": article["url"],
        "text": article["text"]
    })

print(f'Документов всего: {len(documents)}')

Документов всего: 37


In [41]:
def split_text(text, chunk_size=1000, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap
    return chunks

chunks = []

for doc_id, doc in enumerate(documents):
    doc_chunks = split_text(doc["text"])

    for chunk_id, chunk_text in enumerate(doc_chunks):
        chunks.append({
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "title": doc["title"],
            "url": doc["url"],
            "text": chunk_text
        })

print(f'Всего чанков: {len(chunks)}')

Всего чанков: 164


In [42]:
chunk_texts = [chunk["text"] for chunk in chunks]
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=50000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf_vectorizer.fit_transform(chunk_texts)
print(tfidf_matrix.shape)

(164, 16078)


In [43]:
def sparse_search(query, top_k=5):
    query_vec = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_indices = scores.argsort()[::-1][:top_k]
    results = []
    for id_chunk in top_indices:
        item = chunks[id_chunk].copy()
        item['sparse_score'] = float(scores[id_chunk])
        results.append(item)
    return results

In [44]:
results = sparse_search("Как добавить товар в акцию?", top_k=3)

for r in results:
    print(r['title'])
    print(r["url"])
    print('score:', r["sparse_score"])
    print(r["text"][:500])
    print('-' * 100)

Добавление товаров в акции
https://academy.lamoda.ru/articles/aktsii/kak-dobavit-tovary-v-aktsiyu-cherez-interfeys/
score: 0.17022752085459075
Добавить товары в акцию можно двумя способами:

через интерфейс — для этого нажмите кнопку «Добавить товары»

с помощью XLSX файла

— для этого нажмите кнопку «Добавить товары через Excel»

Вам будет предложено добавить товары, которые уже прошли фильтрацию по минимальной цене с учетом скидки по акции и категориям, доступным для участия, вне зависимости от того, есть товар на стоке или нет.

Добавить товары в акцию можно в любой момент до окончания акции.

После окончания акции цена продажи това
----------------------------------------------------------------------------------------------------
Инструкция по работе с единым Excel-шаблоном в акциях
https://academy.lamoda.ru/articles/aktsii/instruktsiya-po-rabote-s-edinym-excel-shablonom-v-aktsiyakh/
score: 0.12543143818643757
Две колонки доступны для редактирования:

«Скидка для участия в акции»


In [45]:
embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")
dense_texts = ["passage: " + chunk["text"] for chunk in chunks]

dense_embeddings = embedding_model.encode(
    dense_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(dense_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

(164, 384)


Для dense retrieval я использую модель `intfloat/multilingual-e5-small`, потому что она поддерживает русский язык, предназначена для retrieval сценариев и достаточно лёгкая для запуска в google colab. Для повышения качества можно заменить её на `intfloat/multilingual-e5-base`, `intfloat/multilingual-e5-large` или `BAAI/bge-m3`, но это увеличит время индексации и требования к памяти

In [46]:
def dense_search(query, top_k=5):
    query_embedding = embedding_model.encode(["query: " + query], normalize_embeddings=True)[0]

    scores = dense_embeddings @ query_embedding

    top_indices = scores.argsort()[::-1][:top_k]

    results = []

    for id_top_chunks in top_indices:
        item = chunks[id_top_chunks].copy()
        item["dense_score"] = float(scores[id_top_chunks])
        results.append(item)

    return results

In [47]:
results = dense_search("Как добавить товар в акцию?", top_k=3)

for r in results:
    print(r["title"])
    print(r["url"])
    print('score:', r['dense_score'])
    print(r['text'][:500])
    print('-' * 100)

Добавление товаров в акции
https://academy.lamoda.ru/articles/aktsii/kak-dobavit-tovary-v-aktsiyu-cherez-interfeys/
score: 0.9200969338417053
Добавить товары в акцию можно двумя способами:

через интерфейс — для этого нажмите кнопку «Добавить товары»

с помощью XLSX файла

— для этого нажмите кнопку «Добавить товары через Excel»

Вам будет предложено добавить товары, которые уже прошли фильтрацию по минимальной цене с учетом скидки по акции и категориям, доступным для участия, вне зависимости от того, есть товар на стоке или нет.

Добавить товары в акцию можно в любой момент до окончания акции.

После окончания акции цена продажи това
----------------------------------------------------------------------------------------------------
Товары с потенциалом улучшения метрик за счет участия в акциях
https://academy.lamoda.ru/articles/aktsii/tovary-s-potentsialom-uluchsheniya-metrik-za-schet-uchastiya-v-aktsiyakh/
score: 0.9104573130607605
товаров можно скачать в формате Excel.

В файле буд

In [48]:
def dict_for_merge_res(merged, made_result):
    for position_of_res, found_chunk in enumerate(made_result):
        key = (found_chunk["url"], found_chunk["chunk_id"])

        if key not in merged:
            merged[key] = {
                "chunk": found_chunk,
                "score": 0.0
            }

        merged[key]["score"] += 1 / (60 + position_of_res + 1)


def search_knowledge_base(query, top_k=5, candidate_k=20):
    sparse_results = sparse_search(query, top_k=candidate_k)
    dense_results = dense_search(query, top_k=candidate_k)
    merged = {}

    dict_for_merge_res(merged, sparse_results)
    dict_for_merge_res(merged, dense_results)

    ranked = sorted(
        merged.values(),
        key=lambda x: x["score"],
        reverse=True
    )
    results = []
    for row in ranked[:top_k]:
        item = row['chunk'].copy()
        item["hybrid_score"] = row["score"]
        results.append(item)

    return results

In [49]:
results = search_knowledge_base("Как добавить товар в акцию?", top_k=5)

for r in results:
    print(r['title'])
    print(r['url'])
    print('score:', r['hybrid_score'])
    print(r['text'][:700])
    print('*' * 100)

Добавление товаров в акции
https://academy.lamoda.ru/articles/aktsii/kak-dobavit-tovary-v-aktsiyu-cherez-interfeys/
score: 0.03278688524590164
Добавить товары в акцию можно двумя способами:

через интерфейс — для этого нажмите кнопку «Добавить товары»

с помощью XLSX файла

— для этого нажмите кнопку «Добавить товары через Excel»

Вам будет предложено добавить товары, которые уже прошли фильтрацию по минимальной цене с учетом скидки по акции и категориям, доступным для участия, вне зависимости от того, есть товар на стоке или нет.

Добавить товары в акцию можно в любой момент до окончания акции.

После окончания акции цена продажи товара автоматически вернется к своему доакционному значению.

Чтобы добавить товары в акцию через интерфейс, нажмите кнопку «Добавить товары».

Далее выберите товары, которые хотите добавить в акцию, с п
****************************************************************************************************
Товары с потенциалом улучшения метрик за счет участия в

????? ???????? ???????? ????? ??????????? OpenRouter API key. ???? ???????? ???????????? ????? `getpass`, ??????? ?? ?? ??????????? ? ????????.

In [51]:
from google.colab import userdata

api_key = getpass.getpass("Введите OpenRouter API key:")

In [52]:
from openai import OpenAI

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

In [53]:
def format_search_results(results):
    blocks = []

    for id, res in enumerate(results, start=1):
        block = '\n'.join([
            f"[{id}] {res['title']}",
            f'URL: {res['url']}',
            "Фрагмент:",
            res['text'][:1500],
        ])
        blocks.append(block)

    return "\n\n".join(blocks)

In [54]:
SYSTEM_PROMPT = """
Ты консультант по базе знаний Lamoda Seller Academy.

У тебя есть инструмент search_knowledge_base(query)

Твои первые же действия:
1. Сначала всегда вызови search_knowledge_base для вопроса пользователя.
2. Отвечай только на основе найденных фрагментов.
3. Если информации в найденных фрагментах недостаточно, честно скажи, что в базе знаний не найдено достаточно данных.
4. В финальном ответе обязательно укажи ссылки на использованные источники.
5. Отвечай на русском языке.

Формат работы:
Thought: кратко подумай, что нужно найти
Action: search_knowledge_base
Action Input: поисковый запрос
Observation: результаты поиска
Final Answer: итоговый ответ пользователю со ссылками
"""

Я использую react-подобный формат thought/action/observation/final answer, тк это текстовый протокол, который задаёт агенту последовательность. сначала выбрать действие поиска, затем получить результаты и только потом сформировать финальный ответ

In [55]:
def clean_agent_answer(answer):
    return answer.replace("Final Answer:", "").strip()

In [58]:
def react_agent_answer(question, top_k=5, model="openai/gpt-4o-mini"):
    thought = "Нужно найти релевантные статьи в базе знаний Lamoda."
    action = "search_knowledge_base"
    action_input = question

    observation = format_search_results(
        search_knowledge_base(action_input, top_k=top_k)
    )

    final_prompt = f"""
Вопрос пользователя:
{question}

Thought: {thought}
Action: {action}
Action Input: {action_input}
Observation:
{observation}

Сформируй Final Answer на русском языке. Используй только Observation и укажи ссылки.
"""

    response = client.chat.completions.create(
        model=model,
        messages= [
            {'role': 'system', "content": SYSTEM_PROMPT},
            {'role': 'user', "content": final_prompt}
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content

In [59]:
answer = react_agent_answer("Как добавить товар в акцию?")
print(answer)

Final Answer: Чтобы добавить товар в акцию, вы можете воспользоваться двумя способами:

1. **Через интерфейс**:
   - Нажмите кнопку «Добавить товары».
   - Выберите товары с помощью чекбоксов. Для удобства используйте фильтры.
   - Нажмите «Добавить товары» для добавления всех подходящих товаров или выберите конкретные товары и затем нажмите «Добавить товары».

2. **С помощью XLSX файла**:
   - Нажмите кнопку «Добавить товары через Excel».
   - Заполните файл, указав скидку и отметив, что товар участвует в акции (укажите «ДА» в соответствующей колонке).
   - Сохраните файл и загрузите его обратно в систему через окно «Экспорт и импорт».

После добавления товаров вы получите отчет о том, сколько товаров было успешно добавлено и по каким товарам произошли ошибки. 

Дополнительные детали можно найти по следующим ссылкам:
- [Добавление товаров в акции](https://academy.lamoda.ru/articles/aktsii/kak-dobavit-tovary-v-aktsiyu-cherez-interfeys/)
- [Инструкция по работе с единым Excel-шаблоном в

In [ ]:
def print_unique_sources(results):
    seen_urls = set()
    source_number = 1

    for item in results:
        url = item["url"]

        if url in seen_urls:
            continue
        seen_urls.add(url)
        print(f'{source_number}. {item['title']}')
        print(f'   {url}')

        source_number += 1

In [ ]:
test_questions = [
    "Как добавить товар в акцию?",
    "Как зарегистрироваться в Lamoda Seller?"
]

for question in test_questions:
    print("=" * 100)
    print("Вопрос:", question)
    print()

    retrieved = search_knowledge_base(question, top_k=5)

    print("Найденные источники:")
    print_unique_sources(retrieved)

    print()
    print("Ответ агента:")
    print(clean_agent_answer(react_agent_answer(question)))
    print()

## Обоснование решения

### Логика парсинга

Парсер начинает обход с главной страницы Lamoda seller academy. Сначала собираются ссылки на разделы, затем из каждого раздела извлекаются ссылки на страницы внутри `/articles/`, для каждой страницы выполняется попытка извлечь основной текст статьи из html контейнера с текстом статьи. Если страница не содержит ожидаемый блок или не является полноценной статьей, она пропускается и сохраняется в списке `failed_urls`.
Статьи сохраняются в markdown формате: заголовок, ссылка на источник и очищенный текст статьи. Это позволяет использовать их как локальную текстовую базу знаний

### Архитектура retrieval

Я использую гибридный retrieval из двух частей:

1. Sparse retrieval на основе TF-IDF
2. Dense retrieval на основе эмбеддингов `intfloat/multilingual-e5-small`

Sparse приск хорошо находит точные совпадения терминов, например названия разделов, API. Dense поиск помогает находить релевантные фрагменты при переформулированных вопросах пользователя
Результаты sparse и dense retrieval объединяются с помощью reciprocal rank fusion: документы получают вклад в итоговый score в зависимости от позиции в каждом ранжировании

### Дизайн агента

Агент реализован как простой ReAct-цикл без LangGraph, Google ADK и других высокоуровневых фреймворков. На вход он получает вопрос пользователя, затем использует инструмент `search_knowledge_base(query)`, получает релевантные фрагменты базы знаний и формирует финальный ответ. В финальном ответе агент обязан использовать только найденные фрагменты и указывать ссылки на источники

### Ограничения и планы улучшения

Текущая версия делает статический слепок базы знаний и не отслеживает обновления сайта. Часть страниц может быть пропущена, если их html структура отличается от ожидаемой или если контент подгружается динамически

Возможные улучшения:

- улучшить обход сайта и покрытие всех страниц академии
- добавить более аккуратную конвертацию hmtl в markdown
- сделать динамический парсинг сайта
- подобрать размер чанков и overlap на валидационных запросах
- добавить reranking top-K результатов
- дать агенту возможность делать несколько итераций поиска
- добавить тестовый набор вопросов для оценки качества retrieval